# Stooq panel construction

This notebook constructs the universe in the research project. I've downloaded the US stocks data from https://stooq.com/db/h/.

The structure of the notebook is as follows:
- Load data from three different exchanges folder and inspect their structure.
- Parse the data to pandas dataframe and rename columns and tickers.
- Calculate 60 days rolling dollar-volume and rank the top 1500 stocks.
- Save to a parquet file.

In [1]:
from pathlib import Path
import pandas as pd

## Get exchange and path from directories

In [2]:
# base directory for the exchanges
base_dir = Path("data/daily/us")

# choose only stocks directories
exchange_dirs = {
    "NASDAQ": base_dir / "nasdaq stocks",
    "NYSE": base_dir / "nyse stocks",
    "NYSEMKT": base_dir / "nysemkt stocks",
}

all_files = []

# read all the txt files
for exchange, folder in exchange_dirs.items():
    files = list(folder.rglob("*.txt"))
    print(exchange, len(files))
    all_files.extend([(exchange, f) for f in files])

print("Total files:", len(all_files))
all_files[:3]

NASDAQ 4550
NYSE 3610
NYSEMKT 309
Total files: 8469


[('NASDAQ', WindowsPath('data/daily/us/nasdaq stocks/1/aacb.us.txt')),
 ('NASDAQ', WindowsPath('data/daily/us/nasdaq stocks/1/aacbr.us.txt')),
 ('NASDAQ', WindowsPath('data/daily/us/nasdaq stocks/1/aacbu.us.txt'))]

In [3]:
# take sample file and inspect it
sample_file = all_files[0][1]
df_sample = pd.read_csv(sample_file)
df_sample.head()

,<TICKER>,<PER>,<DATE>,<TIME>,<OPEN>,<HIGH>,<LOW>,<CLOSE>,<VOL>,<OPENINT>
0,AACB.US,D,20250407,0,10.10,10.10,9.88,9.880,8365,0
1,AACB.US,D,20250408,0,9.95,9.95,9.95,9.950,144,0
2,AACB.US,D,20250409,0,9.90,9.90,9.90,9.900,125094,0
3,AACB.US,D,20250410,0,9.85,9.90,9.85,9.890,101015,0
4,AACB.US,D,20250411,0,9.88,9.89,9.88,9.885,641672,0


## Parse to dataframe

In [4]:
def parse_stooq_file(filepath, exchange):
    df = pd.read_csv(filepath)

    # change column names
    df.columns = [c.strip().replace("<", "").replace(">", "").lower() for c in df.columns]

    # keep only daily data if present
    if "per" in df.columns:
        df = df[df["per"] == "D"].copy()

    # parse date
    df["date"] = pd.to_datetime(df["date"], format="%Y%m%d", errors="coerce")

    # change ticker names, e.g., from AAPL.US -> AAPL
    df["ticker"] = df["ticker"].astype(str).str.replace(".US", "", regex=False)

    # numeric conversion
    numeric_cols = ["open", "high", "low", "close", "vol"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # rename vol to volume
    df = df.rename(columns={"vol": "volume"})

    # add exchange and source file
    df["exchange"] = exchange
    df["source_file"] = Path(filepath).name

    # keep only needed columns
    keep_cols = ["ticker", "date", "open", "high", "low", "close", "volume", "exchange"]
    df = df[keep_cols].copy()

    # basic cleaning
    df = df.dropna(subset=["ticker", "date", "close", "volume"])
    df = df[df["close"] > 0]
    df = df[df["volume"] >= 0]

    return df

In [5]:
# not all txt files contain historical data
parsed_list = []

for i, (exchange, filepath) in enumerate(all_files, 1):
    if i % 2000 == 0:
        print(f"Processed {i}/{len(all_files)} files")

    try:
        df = parse_stooq_file(filepath, exchange)
        if not df.empty:
            parsed_list.append(df)
    except Exception as e:
        print(f"Failed on {filepath}: {e}")

Failed on data\daily\us\nasdaq stocks\1\aigo.us.txt: No columns to parse from file
Failed on data\daily\us\nasdaq stocks\1\bcacu.us.txt: No columns to parse from file
Failed on data\daily\us\nasdaq stocks\1\capnu.us.txt: No columns to parse from file
Failed on data\daily\us\nasdaq stocks\1\cast.us.txt: No columns to parse from file
Failed on data\daily\us\nasdaq stocks\1\dooo.us.txt: No columns to parse from file
Failed on data\daily\us\nasdaq stocks\1\efty.us.txt: No columns to parse from file
Processed 2000/8469 files
Failed on data\daily\us\nasdaq stocks\2\idacu.us.txt: No columns to parse from file
Failed on data\daily\us\nasdaq stocks\2\mctr.us.txt: No columns to parse from file
Failed on data\daily\us\nasdaq stocks\2\mogo.us.txt: No columns to parse from file
Failed on data\daily\us\nasdaq stocks\2\mrp-w.us.txt: No columns to parse from file
Failed on data\daily\us\nasdaq stocks\2\mtsr.us.txt: No columns to parse from file
Failed on data\daily\us\nasdaq stocks\2\myhb.us.txt: No c

In [6]:
# concatenate into a single prices df
prices = pd.concat(parsed_list, ignore_index=True)
print(prices.shape)
prices.head()

(20336462, 8)


,ticker,date,open,high,low,close,volume,exchange
0,AACB,2025-04-07,10.10,10.10,9.88,9.880,8365.0,NASDAQ
1,AACB,2025-04-08,9.95,9.95,9.95,9.950,144.0,NASDAQ
2,AACB,2025-04-09,9.90,9.90,9.90,9.900,125094.0,NASDAQ
3,AACB,2025-04-10,9.85,9.90,9.85,9.890,101015.0,NASDAQ
4,AACB,2025-04-11,9.88,9.89,9.88,9.885,641672.0,NASDAQ


In [7]:
# remove duplicates and sort by ticker and date
prices = prices.drop_duplicates(subset=["ticker", "date"], keep="last").copy()
prices = prices.sort_values(["ticker", "date"]).reset_index(drop=True)
print(prices.shape)

(20336462, 8)


## Calculate median dollar-volume on 60 days rolling basis and rank stocks monthly

In [8]:
# calculate dollar-volume
prices["dollar_volume"] = prices.close * prices.volume

# calculate median dollar-volume on 60 days rolling basis
prices["median_dollar_volume_60d"] = (
    prices.groupby("ticker")["dollar_volume"]
    .rolling(window=60, min_periods=60)
    .median()
    .reset_index(level=0, drop=True)
)

In [9]:
# create month column
prices["month"] = prices["date"].dt.to_period("M")

# take the median prices from the end of the month
month_end = (
    prices.groupby(["ticker", "month"], as_index=False)
    .tail(1)
    .copy()
)

month_end.head()

,ticker,date,open,high,low,close,volume,exchange,dollar_volume,median_dollar_volume_60d,month
7,A,1999-11-30,27.2844,27.8972,26.5958,27.4110,4.745571e+06,NYSE,1.300808e+08,NaN,1999-11
29,A,1999-12-31,51.6490,51.9359,49.5384,50.2260,2.126347e+06,NYSE,1.067979e+08,NaN,1999-12
49,A,2000-01-31,43.8921,43.9291,42.0636,43.0013,1.145527e+06,NYSE,4.925915e+07,NaN,2000-01
69,A,2000-02-29,65.9020,69.4366,65.8183,67.4805,1.054862e+06,NYSE,7.118264e+07,8.717407e+07,2000-02
92,A,2000-03-31,68.8626,68.8626,58.4711,67.5622,4.110159e+06,NYSE,2.776914e+08,1.057878e+08,2000-03


In [10]:
# drop NaN rows if there are any
month_end = month_end.dropna(subset=("median_dollar_volume_60d")).copy()

# rank by median
month_end["liquid_rank"] = (
    month_end.groupby("month")["median_dollar_volume_60d"]
    .rank(method="first", ascending=False)
)

# take top 1500 liquid stocks
top1500 = month_end[month_end["liquid_rank"] <= 1500].copy()
print(top1500.shape)
top1500.head()

(445308, 12)


,ticker,date,open,high,low,close,volume,exchange,dollar_volume,median_dollar_volume_60d,month,liquid_rank
69,A,2000-02-29,65.9020,69.4366,65.8183,67.4805,1.054862e+06,NYSE,7.118264e+07,8.717407e+07,2000-02,76.0
92,A,2000-03-31,68.8626,68.8626,58.4711,67.5622,4.110159e+06,NYSE,2.776914e+08,1.057878e+08,2000-03,74.0
111,A,2000-04-28,58.4711,58.5089,56.4810,57.5742,1.959183e+06,NYSE,1.127984e+08,1.396292e+08,2000-04,60.0
133,A,2000-05-31,48.0755,50.7980,47.2613,47.8283,4.853778e+06,NYSE,2.321480e+08,1.886477e+08,2000-05,50.0
154,A,2000-06-30,49.8234,51.6490,47.8553,47.9131,9.116944e+06,NYSE,4.368211e+08,2.456412e+08,2000-06,38.0


In [ ]:
# Export as parquet
prices.to_parquet("./parquets/daily_prices_stooq.parquet", index=False)
top1500.to_parquet("./parquets/monthly_top1500_liquidity.parquet", index=False)